# 🌍 Bangla TTS with OmniVoice

**A Google Colab notebook for Bangla Text-to-Speech (TTS), voice cloning, and voice design.**

Explore three speech-generation modes:

- 🎙️ **Voice Cloning** — Generate speech using an authorized reference voice.
- 🎨 **Voice Design** — Create voices with customizable characteristics.
- 🤖 **Auto Voice** — Generate speech with an automatically selected voice.

---

## 🚀 Getting Started

For the best chance of a smooth setup, follow these steps in order:

1. **Select a GPU:** Go to `Runtime → Change runtime type → T4 GPU`.
2. **Start fresh:** Disconnect and delete the existing runtime if you need a clean environment.
3. **Run cells in order:** Execute each cell once.
4. **Restart if prompted:** If Colab displays a *Restart session* button after installation, click it and continue from Cell 2. **Do not rerun the installation cell.**

### ⚠️ Important: Avoid Dependency Conflicts

- Do **not** force-reinstall `transformers` or related dependencies.
- Do **not** install `parler-tts` in this notebook.
- Avoid unnecessary package upgrades that may introduce incompatible versions.

These precautions help reduce dependency conflicts and runtime errors.

---

## 📦 Project Repository

**GitHub:** [ragibcs/bangla-voice-cloner](https://github.com/ragibcs/bangla-voice-cloner)

Visit the repository for the notebook, source code, and project-related information.

---

## 🛡️ Educational Use, Ethics & Disclaimer

> **IMPORTANT — EDUCATIONAL PURPOSE ONLY**

This notebook is intended **strictly for educational, research, and learning purposes**. It demonstrates text-to-speech technology and voice-generation workflows using OmniVoice.

### Responsible Use Policy

- ✅ Use voice cloning only with the speaker's informed consent and appropriate authorization.
- ✅ Respect privacy, copyright, applicable laws, and third-party rights.
- ❌ Do not use this notebook for fraud, scams, deception, harassment, or other illegal activities.
- ❌ Do not impersonate another person or clone their voice without permission.
- ❌ Do not use generated audio to mislead others or violate anyone's rights.

### Disclaimer of Responsibility

This project is provided for educational and research purposes. The repository owner and contributors **do not endorse, encourage, or authorize illegal, deceptive, or abusive use** of this notebook.

Users are solely responsible for their use of the code, generated audio, modifications, and any redistribution. Users must obtain the necessary permissions and comply with applicable laws, model licenses, and third-party rights.

The repository owner and contributors are not responsible for unauthorized or unlawful use by users, to the extent permitted by applicable law.

**Use AI voice technology responsibly. Learn, experiment, and innovate ethically.**

## 1️⃣ Install (run once)

In [ ]:
!pip install -q omnivoice soundfile
print("✅ Install finished. If Colab asks to restart the session, click Restart, then run from the next cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.6 MB/s eta 0:00:00
✅ Install finished. If Colab asks to restart the session, click Restart, then run from the next cell.


## 2️⃣ Environment check

In [ ]:
import sys, importlib
print("Python:", sys.version.split()[0])
for pkg in ["torch", "torchaudio", "transformers", "omnivoice"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:13s}", getattr(m, "__version__", "ok"))
    except Exception as e:
        print(f"{pkg:13s} ❌ {type(e).__name__}: {str(e)[:150]}")

import torch
if not torch.cuda.is_available():
    print("\n⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then reconnect.")
else:
    print("\nGPU:", torch.cuda.get_device_name(0))

Python: 3.13.15
torch         2.11.0+cu130
torchaudio    2.11.0+cu130
transformers  5.18.0
omnivoice     0.2.1

GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## 3️⃣ Load the model

In [ ]:
import re, numpy as np, torch, soundfile as sf
from IPython.display import Audio, display

try:
    from omnivoice import OmniVoice
except Exception as e:
    raise RuntimeError(
        "OmniVoice failed to import (usually a broken transformers install).\n"
        "Fix: Runtime → Disconnect and delete runtime → reconnect → run the install cell once → run again.\n"
        f"Original error: {type(e).__name__}: {e}"
    )

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
print("Loading on", device, "...")

model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
SR = 24000
print("Model loaded ✅")

Loading on cuda:0 ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Model loaded ✅


## 4️⃣ Helper functions

In [ ]:
def split_sentences(text, max_chars=200):
    parts = [p.strip() for p in re.split(r"(?<=[।?!\.])\s+", text.strip()) if p.strip()]
    chunks, cur = [], ""
    for p in parts:
        if len(cur) + len(p) + 1 <= max_chars:
            cur = f"{cur} {p}".strip()
        else:
            if cur: chunks.append(cur)
            cur = p
    if cur: chunks.append(cur)
    return chunks or [text]


def to_numpy(x):
    """Accept torch tensor or numpy array of shape (T,) or (1,T) -> float32 (T,)"""
    if hasattr(x, "detach"):
        x = x.detach().cpu().float().numpy()
    x = np.asarray(x, dtype=np.float32)
    return x.reshape(-1) if x.ndim > 1 else x


def synthesize(text, ref_audio=None, ref_text=None, instruct=None,
               speed=1.0, num_step=32, pause=0.25, seed=42):
    """Generate speech for text of any length. Returns float32 numpy array at 24 kHz."""
    pieces = []
    gap = np.zeros(int(SR * pause), dtype=np.float32)
    for i, chunk in enumerate(split_sentences(text)):
        torch.manual_seed(seed)
        kwargs = dict(text=chunk, speed=speed, num_step=num_step)
        if ref_audio:
            kwargs["ref_audio"] = ref_audio
            if ref_text:
                kwargs["ref_text"] = ref_text
        elif instruct:
            kwargs["instruct"] = instruct
        out = model.generate(**kwargs)
        out = out[0] if isinstance(out, (list, tuple)) else out
        pieces.append(to_numpy(out))
        pieces.append(gap)
    return np.concatenate(pieces[:-1])


def save_and_play(wav, path):
    sf.write(path, wav, SR)          # soundfile: no torchaudio/torchcodec needed
    display(Audio(wav, rate=SR))
    print("Saved:", path)
    return path

## 5️⃣ Voice cloning (best natural tone)

Upload a **clean 5–15 s** clip of one speaker (wav/mp3/aac/m4a all fine — it is converted automatically).

In [12]:
import os, subprocess
from google.colab import files

uploaded = files.upload()
raw_path = list(uploaded.keys())[0]

# Convert to mono 24 kHz WAV, max 15 s -> avoids codec/format errors (e.g. .aac)
ref_audio_path = "reference.wav"
r = subprocess.run(
    ["ffmpeg", "-y", "-i", raw_path, "-t", "15", "-ac", "1", "-ar", "24000", ref_audio_path],
    capture_output=True, text=True,
)
if r.returncode != 0 or not os.path.exists(ref_audio_path):
    raise RuntimeError("Could not convert the audio file:\n" + r.stderr[-500:])

print("Reference ready:", ref_audio_path)
display(Audio(ref_audio_path))

Saving ragibhasanbangla_speech_corpus_v1 Datasets at Hugging Face.wav to ragibhasanbangla_speech_corpus_v1 Datasets at Hugging Face.wav
Reference ready: reference.wav


In [13]:
#@title Cloning settings { run: "auto" }
text = "আসসালামু আলাইকুম! আজকের দিনটা সত্যিই সুন্দর। চলুন, একসাথে বাংলা ভাষার সৌন্দর্য উপভোগ করি। প্রতিহিংসার রাজনীতি করলে কারো অস্তিত্ব থাকতো না" #@param {type:"string"}
ref_text = "" #@param {type:"string"}
speed = 1.0 #@param {type:"slider", min:0.7, max:1.3, step:0.05}
num_step = 32 #@param [16, 24, 32, 48]
pause_between_sentences = 0.25 #@param {type:"number"}
seed = 42 #@param {type:"integer"}

wav = synthesize(text, ref_audio=ref_audio_path, ref_text=ref_text.strip() or None,
                 speed=speed, num_step=num_step, pause=pause_between_sentences, seed=seed)
save_and_play(wav, "omnivoice_clone.wav")

Saved: omnivoice_clone.wav


'omnivoice_clone.wav'

## 6️⃣ Voice design (no reference audio)
Attributes: `male`/`female`, age (`child` … `elderly`), pitch (`very low pitch` … `very high pitch`), `whisper`.

In [14]:
#@title Voice design settings { run: "auto" }
text = "আমি তোমাকে একটা গল্প বলব। অনেক দিন আগের কথা, এক ছোট্ট গ্রামে একটি মেয়ে থাকত।" #@param {type:"string"}
instruct = "female, young adult, moderate pitch" #@param {type:"string"}
speed = 0.95 #@param {type:"slider", min:0.7, max:1.3, step:0.05}

wav = synthesize(text, instruct=instruct, speed=speed)
save_and_play(wav, "omnivoice_design.wav")

Saved: omnivoice_design.wav


'omnivoice_design.wav'

## 7️⃣ Auto voice

In [15]:
wav = synthesize("এটি একটি স্বয়ংক্রিয় কণ্ঠস্বরের পরীক্ষা।")
save_and_play(wav, "omnivoice_auto.wav")

Saved: omnivoice_auto.wav


'omnivoice_auto.wav'

## 8️⃣ Download

In [16]:
import os
from google.colab import files
for f in ["omnivoice_clone.wav", "omnivoice_design.wav", "omnivoice_auto.wav"]:
    if os.path.exists(f):
        files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🛠️ Troubleshooting
| Error | Fix |
|---|---|
| `cannot import name 'PreTrainedConfig' ...` / `Failed to import transformers...` | Mixed `transformers` versions. **Runtime → Disconnect and delete runtime**, then run the install cell once. |
| `NameError: name 'model' is not defined` | Cell 3 failed or wasn't run. Fix its error first. |
| `CUDA out of memory` | Restart runtime; keep `num_step` ≤ 32 and text short. |
| Audio sounds off | Use a cleaner reference clip and type its exact transcript in `ref_text`. |